**Practical Task — Add Conversation Memory to the Chatbot**

*The chatbot should remember what the user said earlier in the same session. This transforms your stateless chatbot into a stateful conversational assistant.*

**What to implement:**

- Assign a unique session ID to each conversation
- Store conversation history using ConversationBufferMemory or equivalent
- Pass history into each prompt so the LLM has context
- Test multi-turn conversation: tell it your name, then ask it what your name is
- Implement window memory to limit history to the last 5 messages
- Print the full conversation history after each turn for debugging

In [2]:
"""
===========================================================
Practical Task - Chatbot with Conversation Memory (LCEL)

Features:
1. Uses Groq Llama 3.3 70B Versatile
2. Uses ChatPromptTemplate
3. Uses MessagesPlaceholder
4. Uses StrOutputParser
5. Uses LCEL (prompt | llm | parser)
6. Uses RunnableWithMessageHistory
7. Maintains unique session id
8. Keeps only the last 10 messages in memory
9. Prints conversation history after every turn
===========================================================
"""

import uuid

from langchain_groq import ChatGroq

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)

from langchain_core.output_parsers import StrOutputParser

from langchain_core.chat_history import InMemoryChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory
import os
from dotenv import load_dotenv

load_dotenv()


# ===========================================================
# Create Groq LLM
# ===========================================================
llm = ChatGroq(
    model=os.getenv("CHAT_MODEL_NAME"),
    temperature=0
)


# ===========================================================
# Create Prompt Template
# MessagesPlaceholder automatically injects the chat history
# ===========================================================
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful AI assistant. Answer naturally using the conversation history."
        ),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)


# ===========================================================
# Output Parser
# Converts AIMessage into plain string
# ===========================================================
parser = StrOutputParser()


# ===========================================================
# Create LCEL Chain
# prompt -> llm -> parser
# ===========================================================
chain = prompt | llm | parser


# ===========================================================
# Dictionary to store all conversation sessions
# Key   -> session_id
# Value -> ChatMessageHistory
# ===========================================================
store = {}


# ===========================================================
# Function to get chat history for a session
# If session does not exist, create one.
# ===========================================================
def get_session_history(session_id: str):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    history = store[session_id]

    # -------------------------------------------------------
    # Window Memory
    # Keep only the last 10 messages
    # -------------------------------------------------------
    if len(history.messages) > 10:
        history.messages = history.messages[-10:]

    return history


# ===========================================================
# Wrap the LCEL chain with message history
# ===========================================================
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)


# ===========================================================
# Generate a unique session id
# ===========================================================
session_id = str(uuid.uuid4())

print("=" * 60)
print("Chatbot Started")
print(f"Session ID : {session_id}")
print("Type 'exit' to quit.")
print("=" * 60)


# ===========================================================
# Chat Loop
# ===========================================================
while True:

    user_input = input("\nYou : ")

    if user_input.lower() == "exit":
        print("\nGoodbye!")
        break

    # -------------------------------------------------------
    # Invoke chatbot
    # -------------------------------------------------------
    response = chatbot.invoke(
        {"input": user_input},
        config={
            "configurable": {
                "session_id": session_id
            }
        },
    )

    print(f"AI  : {response}")

    # -------------------------------------------------------
    # Print Conversation History
    # -------------------------------------------------------
    history = get_session_history(session_id)

    print("\n------ Conversation History ------")

    for i, message in enumerate(history.messages, start=1):

        print(f"{i}. {message.type.upper()} : {message.content}")

    print("-" * 35)

Chatbot Started
Session ID : 957155a3-a1b7-4fd0-b718-9d53d3a0aefe
Type 'exit' to quit.


d:\AI\langchain-roadmap\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AI  : The capital of India is New Delhi.

------ Conversation History ------
1. HUMAN : capital of india
2. AI : The capital of India is New Delhi.
-----------------------------------
AI  : Nice to meet you, Sahil. Is there something I can help you with or would you like to chat?

------ Conversation History ------
1. HUMAN : capital of india
2. AI : The capital of India is New Delhi.
3. HUMAN : my name is sahil
4. AI : Nice to meet you, Sahil. Is there something I can help you with or would you like to chat?
-----------------------------------
AI  : So, Sahil, you're 30 years old. That's a great age, often considered a time of stability and growth. What do you do, if you don't mind me asking?

------ Conversation History ------
1. HUMAN : capital of india
2. AI : The capital of India is New Delhi.
3. HUMAN : my name is sahil
4. AI : Nice to meet you, Sahil. Is there something I can help you with or would you like to chat?
5. HUMAN : my age is 30
6. AI : So, Sahil, you're 30 years old.